#### Problem 7

a. Estimate the proportion of homes with central air.

In [3]:
import numpy as np
import pandas as pd

df = pd.read_csv("../data/ames_prices.csv")

df["has_central_air"] = (df["Central.Air"] == "Y").astype(int)
original_proportion = df["has_central_air"].mean()

print(f"Proportion with central air: {original_proportion:.4f}")


Proportion with central air: 0.9331


b. Using a reproducible random-number seed, generate at least 2,000 bootstrap estimates of this proportion. Report the seed and number of bootstrap replications.

In [4]:
seed = 100
n_bootstrap = 2000

rng = np.random.default_rng(seed)

central_air = df["has_central_air"].to_numpy()
n_homes = len(central_air)

bootstrap_proportions = []

for _ in range(n_bootstrap):
    sample_indices = rng.integers(0, n_homes, size=n_homes)
    bootstrap_sample = central_air[sample_indices]
    bootstrap_proportions.append(bootstrap_sample.mean())

bootstrap_proportions = np.array(bootstrap_proportions)

print(f"Seed: {seed}")
print(f"Bootstrap replications: {n_bootstrap}")

Seed: 100
Bootstrap replications: 2000


c. Compute the bootstrap standard error as the sample standard deviation of the bootstrap estimates (using ddof=1 ensures that there's the 1/(N −1) degrees of freedom correction).

In [6]:
bootstrap_se = np.std(bootstrap_proportions, ddof=1)

print(f"Bootstrap SE: {bootstrap_se:.6f}")

Bootstrap SE: 0.004598


d. Using the original estimate and the bootstrap standard error, construct Normal 90%, 95%, and 99% confidence intervals

In [7]:
critical_values = {
    0.90: 1.645,
    0.95: 1.96,
    0.99: 2.576
}

confidence_intervals = {}

for confidence, critical_value in critical_values.items():
    lower = original_proportion - critical_value * bootstrap_se
    upper = original_proportion + critical_value * bootstrap_se

    confidence_intervals[confidence] = (lower, upper)

    print(f"{confidence:.0%} CI: [{lower:.4f}, {upper:.4f}]")

90% CI: [0.9255, 0.9407]
95% CI: [0.9241, 0.9421]
99% CI: [0.9213, 0.9450]


e. Use each confidence interval to evaluate $H_0: p = 0.90$. At the matching significance levels $\alpha = 0.10, 0.05, 0.01$, state whether you reject or fail to reject the null hypothesis. Briefly interpret your results.

In [8]:
null_proportion = 0.90

for confidence, (lower, upper) in confidence_intervals.items():
    alpha = 1 - confidence

    if lower <= null_proportion <= upper:
        decision = "Fail to reject H0"
    else:
        decision = "Reject H0"

    print(f"Alpha = {alpha:.2f}: {decision}")


Alpha = 0.10: Reject H0
Alpha = 0.05: Reject H0
Alpha = 0.01: Reject H0


Our results make sense. All of our confidence intervals range from approximately 92% to 95%. Since 0.90 is not a plausible value within any of these intervals, we reject the null hypothesis H0: p = 0.90 at all three significance levels.